# AEW43003 NP-UTR PyMOL Render Controls

Editable rendering workflow for the Boltz 2 AEW43003.1 nucleoprotein monomer bound to the JN542542.1 5-prime UTR. The model used here is the RBD-pocket-conditioned run with the 36-nt UTR from bases 1-36 before the CDS start at base 37.

The cells below only control rendering and cropping. They do not rerun Boltz.

In [ ]:
from pathlib import Path
import json
import subprocess
from PIL import Image, ImageChops
from IPython.display import Image as DisplayImage, display

CANDIDATES = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in CANDIDATES if (path / "data" / "processed").exists())
PYMOL_BIN = Path('/opt/homebrew/Caskroom/miniconda/base/bin/pymol')

RUN_DIR = PROJECT_ROOT / 'data/processed/structures/aew43003_monomer_utr_rbd_pocket_conditioned'
PRED_DIR = RUN_DIR / 'outputs/boltz_results_aew43003_monomer_utr_rbd_pocket_conditioned/predictions/aew43003_monomer_utr_rbd_pocket_conditioned'
CIF_PATH = PRED_DIR / 'aew43003_monomer_utr_rbd_pocket_conditioned_model_0.cif'
CONFIDENCE_JSON = PRED_DIR / 'confidence_aew43003_monomer_utr_rbd_pocket_conditioned_model_0.json'

assert CIF_PATH.exists(), CIF_PATH
assert PYMOL_BIN.exists(), PYMOL_BIN

In [ ]:
with CONFIDENCE_JSON.open() as handle:
    confidence = json.load(handle)

{k: confidence[k] for k in ['confidence_score', 'ptm', 'iptm', 'complex_plddt', 'complex_ipde']}

## Edit Render Settings

Change these values, then rerun the cells below. The render uses no figure title text.

Color tuples can be either `0-1` floats or `0-255` RGB integers. The default render is transparent; the optional preview background is only for checking contrast inside the notebook.

In [ ]:
settings = {
    'stem': 'aew43003_monomer_utr_muted_transparent',
    'width': 2200,
    'height': 1500,
    'dpi': 600,
    'protein_chain': 'A',
    'rna_chain': 'B',

    # Muted colors closer to the reference schematic. 0-255 RGB is accepted.
    'protein_color_rgb': (150, 170, 82),
    'rna_color_rgb': (198, 91, 123),
    'preview_background_rgb': (210, 232, 226),

    'surface_transparency': 0.00,
    'rna_tube_radius': 0.18,
    'rna_stick_radius': 0.09,

    # View controls.
    'turn_x': -10,
    'turn_y': 28,
    'turn_z': -5,
    'zoom_buffer': 6,
    'crop_pad': 120,

    # Lighting controls. Lower ambient and specular avoid the washed-out neon look.
    'ray_trace_mode': 1,
    'ray_shadows': 'off',
    'ambient': 0.48,
    'direct': 0.58,
    'specular': 0.06,
    'shininess': 10,
    'reflect': 0.10,
}
settings

In [ ]:
def normalize_rgb(values):
    vals = tuple(float(v) for v in values)
    if max(vals) > 1.0:
        vals = tuple(v / 255.0 for v in vals)
    return vals


def rgb_tuple(values):
    vals = normalize_rgb(values)
    return '[' + ', '.join(f'{v:.4f}' for v in vals) + ']'


def build_pymol_script(settings):
    stem = settings['stem']
    transparent_png = RUN_DIR / f'{stem}_transparent.png'
    pml_path = RUN_DIR / f'{stem}.pml'

    ray_shadows = settings.get('ray_shadows', 'off')
    script = f'''reinitialize
load {CIF_PATH}, np_utr

set bg_rgb, white
set opaque_background, off
set ray_opaque_background, off
set antialias, 2
set ray_trace_mode, {settings['ray_trace_mode']}
set ray_shadows, {ray_shadows}
set ambient, {settings['ambient']}
set direct, {settings['direct']}
set specular, {settings['specular']}
set shininess, {settings['shininess']}
set reflect, {settings['reflect']}
set depth_cue, off
set orthoscopic, on
set field_of_view, 18
set valence, off
set surface_quality, 1
set surface_smooth_edges, on
set ray_trace_fog, 0

set_color np_pastel_olive, {rgb_tuple(settings['protein_color_rgb'])}
set_color rna_muted_rose, {rgb_tuple(settings['rna_color_rgb'])}

hide everything
show surface, chain {settings['protein_chain']}
color np_pastel_olive, chain {settings['protein_chain']}
set transparency, {settings['surface_transparency']}, chain {settings['protein_chain']}

show cartoon, chain {settings['rna_chain']}
show sticks, chain {settings['rna_chain']}
color rna_muted_rose, chain {settings['rna_chain']}
set cartoon_ring_mode, 3
set cartoon_ladder_mode, 1
set cartoon_nucleic_acid_mode, 4
set cartoon_tube_radius, {settings['rna_tube_radius']}, chain {settings['rna_chain']}
set stick_radius, {settings['rna_stick_radius']}, chain {settings['rna_chain']}

orient np_utr
turn x, {settings['turn_x']}
turn y, {settings['turn_y']}
turn z, {settings['turn_z']}
zoom np_utr, {settings['zoom_buffer']}

ray {settings['width']}, {settings['height']}
png {transparent_png}, dpi={settings['dpi']}
'''
    pml_path.write_text(script)
    return pml_path, transparent_png

pml_path, transparent_png = build_pymol_script(settings)
pml_path

In [ ]:
def run_pymol(pml_path):
    subprocess.run([str(PYMOL_BIN), '-cq', str(pml_path)], check=True, cwd=PROJECT_ROOT)

run_pymol(pml_path)
transparent_png

In [ ]:
def crop_to_content(path, pad=120):
    path = Path(path)
    img = Image.open(path).convert('RGBA')
    bbox = img.getchannel('A').getbbox()
    if bbox is None:
        raise ValueError(f'No visible alpha content detected in {path}')

    left = max(0, bbox[0] - pad)
    top = max(0, bbox[1] - pad)
    right = min(img.width, bbox[2] + pad)
    bottom = min(img.height, bbox[3] + pad)
    crop = img.crop((left, top, right, bottom))
    out = path.with_name(path.stem + '_cropped.png')
    crop.save(out)
    return out


def make_preview_on_background(path, background_rgb, suffix='_preview_bg'):
    path = Path(path)
    img = Image.open(path).convert('RGBA')
    rgb = tuple(round(v * 255) for v in normalize_rgb(background_rgb))
    bg = Image.new('RGBA', img.size, rgb + (255,))
    bg.alpha_composite(img)
    out = path.with_name(path.stem + suffix + '.png')
    bg.save(out)
    return out

transparent_cropped = crop_to_content(transparent_png, settings['crop_pad'])
preview_cropped = make_preview_on_background(transparent_cropped, settings['preview_background_rgb'])
transparent_cropped, preview_cropped

In [ ]:
# The first display is the true transparent PNG. The second is only a preview composited in Python.
display(DisplayImage(filename=str(transparent_cropped)))
display(DisplayImage(filename=str(preview_cropped)))

## Useful Variants

Copy one of these small setting updates into the settings cell above if you want a different view.

In [ ]:
# More RNA-forward view
rna_forward = dict(settings, stem='aew43003_monomer_utr_rna_forward_muted', turn_x=-18, turn_y=44, turn_z=-8, zoom_buffer=7)

# Slightly darker, closer to the small reference image
reference_like = dict(
    settings,
    stem='aew43003_monomer_utr_reference_like',
    protein_color_rgb=(137, 158, 77),
    rna_color_rgb=(190, 83, 116),
    ambient=0.42,
    direct=0.62,
    specular=0.04,
    reflect=0.08,
)

# Transparent-background asset for Illustrator/PowerPoint compositing
transparent_asset = dict(settings, stem='aew43003_monomer_utr_transparent_asset')